## Create Figures for Wilson et al., 2026
Figure Generation for Wilson, K., Lott, S.G., Anarde, K., 2026.  Awards, Keynotes, and Gender Equity in Coastal Geoscience and Engineering: A Fifty-Year Perspective, Cambridge Prisms: Coastal Futures. 

Workflow:
* Open csv as dataframe
* Clean dataframe to be useable columns and combine organizations
* Create functions for plot creation: Add white legend, prep data for each plot, create plots
* Create and save plots

In [ ]:
import altair as alt
import pandas as pd

In [ ]:
df = pd.read_csv('SM1_AwardDemographics_AllData_csv.csv', encoding='cp1252', skiprows=8)
df = df.drop(df.columns[11:22], axis=1)
print(df.columns)

In [ ]:
# Rename columns for easier analysis
df = df.rename(columns={'Organization or Professional Society': 'Organization',
    'Career Stage': 'CareerStage',
    'Honor Type': 'HonorType'
})

df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df = df.dropna(subset=['Year'])
df['Year'] = df['Year'].astype(int)
df = df[df['Year'] >= 1978]

# Filter to only include rows that are in analysis
df = df[df['Included in Wilson et al., Analysis '] == 1]

In [ ]:
# Define the three ACS organizations to combine
acs_variations = [
    'Australian Coastal Society Coast to Coast',
    'Australian Coastal Society New South Wales Coastal Conference',
    'Australian Coastal Society Queensland Coastal Conference'
]

# Replace all three with "ACS"
acs_mapping = {var: 'ACS' for var in acs_variations}
df['Organization'] = df['Organization'].replace(acs_mapping)

In [ ]:
# Separate awards and keynotes based on Honor Type
awards_df = df[df['HonorType'] == 'Award'].copy()
keynote_df = df[df['HonorType'] == 'Keynote + Plenary'].copy()

print(len(df))
print(f"Number of Awards: {len(awards_df)}")
print(f"Number of Keynotes + Plenaries: {len(keynote_df)}")
print(f"Years range: {df['Year'].min()} - {df['Year'].max()}")
print(f" Number of Organizations: {df['Organization'].nunique()}")

In [ ]:
# Functions for creating plots
def add_white_legend(plot, text="Award given to another gender"):
    """Add white box annotation for zero-count cells that had events"""
    legend_data = pd.DataFrame({
        'text': [text],
        'x': [0],
        'y': [0]
    })
    
    white_rect = alt.Chart(legend_data).mark_rect(
        width=30,
        height=20,
        stroke='black',
        strokeWidth=1,
        fill='white'
    ).encode(
        x=alt.value(850),
        y=alt.value(250)
    )
    
    white_text = alt.Chart(legend_data).mark_text(
        align='left',
        baseline='middle',
        dx=25,
        fontSize=14
    ).encode(
        x=alt.value(850),
        y=alt.value(250),
        text='text:N'
    )

    return alt.layer(plot, white_rect, white_text)

def prep_gender_percentages(df, group_col, order=None):
    """Prepare percentage data for diverging bar charts"""
    # Calculate total counts per group
    totals = df.groupby(group_col)[["Male", "Female+"]].sum().sum(axis=1)
    
    # Calculate percentages
    counts = df.groupby(group_col)[["Male", "Female+"]].sum()
    percentages = counts.div(totals, axis=0) * 100
    
    # Convert male percentages to negative
    percentages["Male"] = -percentages["Male"]
    
    # Melt to long format
    percentages = percentages.reset_index()
    percentages = percentages.melt(id_vars=group_col, 
                                  value_vars=["Male", "Female+"],
                                  var_name="Gender", 
                                  value_name="Percentage")
    
    # Add order if specified
    if order:
        percentages[group_col] = pd.Categorical(percentages[group_col], categories=order, ordered=True)
        percentages = percentages.sort_values(group_col)
    
    return percentages

def create_heatmap_data(df, org_col='Organization'):
    """Prepare heatmap data for awards/keynotes by organization and year"""
    # Create complete grid
    all_orgs = df[org_col].unique()
    all_years = df['Year'].unique()
    complete_grid = pd.MultiIndex.from_product([all_orgs, all_years], names=['Organization', 'Year']).to_frame(index=False)
    
    # When were awards/keynotes given?
    org_year_with_events = df.groupby([org_col, 'Year']).size().reset_index(name='AnyEvent')
    org_year_with_events['EventGiven'] = True
    complete_grid = complete_grid.merge(org_year_with_events[[org_col, 'Year', 'EventGiven']], 
                                       on=['Organization', 'Year'], how='left')
    complete_grid['EventGiven'] = complete_grid['EventGiven'].fillna(False)
    
    # Process by gender using Female+ column
    gender_data = df.melt(
        id_vars=['Year', org_col, 'CareerStage', 'Award', 'Name'],
        value_vars=['Male', 'Female+'],
        var_name='Gender',
        value_name='Count'
    )
    gender_data = gender_data[gender_data['Count'] > 0]
    heatmap_data = gender_data.groupby(['Organization', 'Year', 'Gender']).size().reset_index(name='EventCount')
    
    # Split by gender
    male_data = heatmap_data[heatmap_data['Gender'] == 'Male'].copy()
    female_plus_data = heatmap_data[heatmap_data['Gender'] == 'Female+'].copy()
    
    # Create final datasets
    male_complete = complete_grid.merge(male_data[['Organization', 'Year', 'EventCount']], 
                                       on=['Organization', 'Year'], how='left')
    male_complete['EventCount'] = male_complete['EventCount'].fillna(0)
    female_complete = complete_grid.merge(female_plus_data[['Organization', 'Year', 'EventCount']], 
                                         on=['Organization', 'Year'], how='left')
    female_complete['EventCount'] = female_complete['EventCount'].fillna(0)
    
    # Create display values: -1 for no event, 0 for event with no people of that gender
    male_complete['DisplayValue'] = male_complete.apply(
        lambda row: -1 if not row['EventGiven'] else (0 if row['EventCount'] == 0 else row['EventCount']), 
        axis=1
    )
    female_complete['DisplayValue'] = female_complete.apply(
        lambda row: -1 if not row['EventGiven'] else (0 if row['EventCount'] == 0 else row['EventCount']), 
        axis=1
    )
    
    # Sort organizations
    org_totals = heatmap_data.groupby('Organization')['EventCount'].sum().reset_index()
    org_totals_sorted = org_totals.sort_values('EventCount', ascending=False)
    organization_order = org_totals_sorted['Organization'].tolist()
    
    return male_complete, female_complete, organization_order

def create_heatmap_plot(male_data, female_data, org_order, male_title, female_title, main_title, legend_text):
    """Create heatmap visualization for awards or keynotes"""
    # Male plot
    male_base = alt.Chart(male_data).encode(
        x=alt.X('Year:O', title='Year', axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('Organization:N', title='Organization', sort=org_order)
    ).properties(width=800, height=400)
    
    male_no_events = male_base.transform_filter(alt.datum.DisplayValue == -1).mark_rect().encode(
        color=alt.value('#cccccc'), 
        tooltip=['Organization', 'Year', alt.Tooltip('EventCount:Q', title='Count')]
    )
    male_zero_events = male_base.transform_filter(alt.datum.DisplayValue == 0).mark_rect().encode(
        color=alt.value('white'), 
        tooltip=['Organization', 'Year', alt.Tooltip('EventCount:Q', title='Count')]
    )
    male_positive_events = male_base.transform_filter(alt.datum.DisplayValue > 0).mark_rect().encode(
        color=alt.Color('EventCount:Q', 
                       scale=alt.Scale(scheme='blues'), 
                       legend=alt.Legend(title="Number")),
        tooltip=['Organization', 'Year', alt.Tooltip('EventCount:Q', title='Count')]
    )
    male_plot = (male_no_events + male_zero_events + male_positive_events).properties(title=male_title)
    
    # Female+ plot
    female_base = alt.Chart(female_data).encode(
        x=alt.X('Year:O', title='Year', axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('Organization:N', title='Organization', sort=org_order)
    ).properties(width=800, height=400)
    
    female_no_events = female_base.transform_filter(alt.datum.DisplayValue == -1).mark_rect().encode(
        color=alt.value('#cccccc'), 
        tooltip=['Organization', 'Year', alt.Tooltip('EventCount:Q', title='Count')]
    )
    female_zero_events = female_base.transform_filter(alt.datum.DisplayValue == 0).mark_rect().encode(
        color=alt.value('white'), 
        tooltip=['Organization', 'Year', alt.Tooltip('EventCount:Q', title='Count')]
    )
    female_positive_events = female_base.transform_filter(alt.datum.DisplayValue > 0).mark_rect().encode(
        color=alt.Color('EventCount:Q', 
                       scale=alt.Scale(scheme='blues'), 
                       legend=alt.Legend(title="Number")),
        tooltip=['Organization', 'Year', alt.Tooltip('EventCount:Q', title='Count')]
    )
    female_plot = (female_no_events + female_zero_events + female_positive_events).properties(title=female_title)
    
    # Add white legend to male plot only
    male_with_legend = add_white_legend(male_plot, legend_text)
    
    # Combine plots
    combined = alt.vconcat(male_with_legend, female_plot).properties(
        title=main_title
    ).resolve_scale(
        x='shared',
        color='shared'
    )
    
    return combined

def create_diverging_percentage_chart(data, group_col, title, ylabel, include_legend_bg=False):
    """Create diverging bar chart showing gender percentages"""
    color_scale = alt.Scale(domain=["Male", "Female+"],
                          range=["#7570b3", "#d95f02"])
    
    max_data_value = max(abs(data['Percentage']))
    buffer = 5
    scale_domain = (-max_data_value - buffer, max_data_value + buffer)
    
    # Create axis values
    axis_values = []
    step = 20
    neg_max = min(100, max_data_value + buffer)
    for i in range(0, int(neg_max) + step, step):
        if i > 0:
            axis_values.append(-i)
    axis_values.append(0)
    pos_max = min(100, max_data_value + buffer)
    for i in range(step, int(pos_max) + step, step):
        axis_values.append(i)
    axis_values.sort()
    
    # Legend background (only for first chart)
    if include_legend_bg:
        legend_background = alt.Chart(pd.DataFrame({'x': [0]})).mark_rect(
            color='white',
            opacity=0.9,
            stroke='lightgray',
            strokeWidth=1,
            cornerRadius=5
        ).encode(
            x=alt.value(305),
            x2=alt.value(375),
            y=alt.value(0),
            y2=alt.value(34)
        )
    else:
        legend_background = alt.Chart(pd.DataFrame()).mark_text().properties(width=0, height=0)
    
    # Create bars
    bars = alt.Chart(data).mark_bar().encode(
        y=alt.Y(f"{group_col}:N", 
               title=ylabel,
               sort=None),
        x=alt.X("Percentage:Q", title="Percentage (%)",
                scale=alt.Scale(domain=scale_domain),
               axis=alt.Axis(
                   values=axis_values,
                   labelExpr="abs(datum.value)"
               )),
        color=alt.Color("Gender:N", 
                      scale=color_scale, 
                      legend=alt.Legend(
                          title="",
                          orient="none",
                          legendX=310,
                          legendY=5,
                          direction="vertical",
                          titleAnchor="middle"
                      )),
        order=alt.Order("Gender_order:Q", sort="ascending")
    ).transform_calculate(
        Gender_order="datum.Gender == 'Male' ? 0 : 1"
    ).properties(
        width=400,
        height=250
    )
    
    # Zero line
    zero_line = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(
        color='black',
        strokeWidth=1
    ).encode(x='x:Q')
    
    # Combine
    if include_legend_bg:
        chart = (legend_background + bars + zero_line).properties(title=title)
    else:
        chart = (bars + zero_line).properties(title=title)
    
    return chart

In [ ]:
# Plot awards heatmap

# Prepare awards heatmap data
awards_male, awards_female, awards_org_order = create_heatmap_data(awards_df)

awards_heatmap = create_heatmap_plot(
    awards_male, awards_female, awards_org_order,
    'Male Awardees',
    'Female+ Awardees',
    'Award Distribution by Organization and Year',
    'Award given to another gender'
)

# Save awards heatmap
awards_heatmap.save('awards-heatmap-highres.png', ppi=300)
awards_heatmap.save('awards-heatmap.svg')

In [ ]:
# Plot keynote speakers heatmap

# Prepare data
keynote_male, keynote_female, keynote_org_order = create_heatmap_data(keynote_df)

keynote_heatmap = create_heatmap_plot(
    keynote_male, keynote_female, keynote_org_order,
    'Male Keynote Speakers',
    'Female+ Keynote Speakers',
    'Keynote Speaker Distribution by Organization and Year',
    'Keynote given by another gender'
)

keynote_heatmap.save('keynote-heatmap-highres.png', ppi=300)
keynote_heatmap.save('keynote-heatmap.svg')

In [ ]:
# Plot diverging percentages by gender identity, career stage, and organization

# Organization (Awards)
org_percentages = prep_gender_percentages(awards_df, "Organization")

# Organization (Keynotes)
org_percentages_keynote = prep_gender_percentages(keynote_df, "Organization")

# Career Stage
print("Career stages in awards data:", sorted(awards_df['CareerStage'].dropna().unique()))
career_order = ["Any", "Late", "Mid", "Early", "Student"]

# Filter to only include career stages that exist in the data
existing_career_stages = [stage for stage in career_order if stage in awards_df['CareerStage'].values]
career_percentages = prep_gender_percentages(
    awards_df[awards_df['CareerStage'].isin(existing_career_stages)], 
    "CareerStage", 
    order=existing_career_stages
)

# Create charts
chart1 = create_diverging_percentage_chart(
    org_percentages, "Organization", 
    "(A) Distribution of Awards by Organization", 
    "Organization", 
    include_legend_bg=True
)

chart2 = create_diverging_percentage_chart(
    career_percentages, "CareerStage", 
    "(B) Distribution of Awards by Career Stage", 
    "Career Stage", 
    include_legend_bg=False
)

chart3 = create_diverging_percentage_chart(
    org_percentages_keynote, "Organization", 
    "(C) Distribution of Keynote Speakers by Organization", 
    "Organization", 
    include_legend_bg=False
)

# Combine charts
final_chart = alt.vconcat(chart1, chart2, chart3).resolve_scale(
    color='shared'
).configure_view(
    strokeWidth=0
)

final_chart.save('awards-keynotes-percentages-ppi300.png', ppi=300)
final_chart.save('awards-keynotes-percentages.svg')

In [ ]:
# Plot time series of awards and gender identities

# Process combined data for time series
def process_for_timeseries(df, source_name):
    """Process dataframe for time series visualization"""
    melted = df.melt(id_vars=['Year'], 
                     value_vars=['Male', 'Female+'],
                     var_name='Gender', 
                     value_name='Count')
    melted['Source'] = source_name
    return melted

# Process both types
awards_processed = process_for_timeseries(awards_df, 'Awards')
keynote_processed = process_for_timeseries(keynote_df, 'Keynotes')

# Combine
combined_df = pd.concat([awards_processed, keynote_processed], ignore_index=True)

# Group by Year and Gender
source = combined_df.groupby(['Year', 'Gender'])['Count'].sum().reset_index()

# Combine into all years and genders
all_years = sorted(df['Year'].unique())
all_genders = ['Male', 'Female+']
year_gender_grid = pd.DataFrame([(year, gender) for year in all_years for gender in all_genders],
                               columns=['Year', 'Gender'])

# Merge with data
source_complete = year_gender_grid.merge(source, on=['Year', 'Gender'], how='left')
source_complete['Count'] = source_complete['Count'].fillna(0)

# Add stack order
source_complete['stack_order'] = source_complete['Gender'].map({
    'Male': 1,
    'Female+': 2
})

# Define colors
gender_colors = ["#7570b3", "#d95f02"]

# Create legend background
legend_background = alt.Chart(pd.DataFrame({'x': [0]})).mark_rect(
    color='white',
    opacity=0.9,
    stroke='lightgray',
    strokeWidth=1,
    cornerRadius=5
).encode(
    x=alt.value(10),
    x2=alt.value(125),
    y=alt.value(3),
    y2=alt.value(55)
)

# Create bars
bars = alt.Chart(source_complete, width=alt.Step(100)).mark_bar(size=22).encode(
    x=alt.X('Year:O', 
           axis=alt.Axis(title='Year', labelAngle=45),
           scale=alt.Scale(padding=0.5, paddingInner=0.8)),
    y=alt.Y('sum(Count):Q', 
           title='Number of Awards and Keynotes Given',
           scale=alt.Scale(domain=[0, max(source_complete['Count']) * 1.5])),
    color=alt.Color(
        'Gender:N',
        scale=alt.Scale(
            domain=['Male', 'Female+'],
            range=gender_colors
        ),
        legend=alt.Legend(
            title="",
            orient="none",
            legendX=15,
            legendY=10,
            direction="vertical",
            titleAnchor="middle"
        )
    ),
    order=alt.Order('stack_order:Q')
).properties(
    width=1100,
    height=700,
    title="Awards and Keynotes Given over Time"
)

# Combine
time_series_chart = (legend_background + bars).properties(
    padding=20
).configure_view(
    strokeWidth=0
).configure(autosize=alt.AutoSizeParams(resize=True)).configure_axis(
    labelFontSize=18,
    titleFontSize=24
).configure_legend( 
    labelFontSize=18  
).configure_title(
    fontSize=30
)

time_series_chart.save('awards_keynotes_timeseries_ppi300.png', ppi=300)
final_chart.save('awards-keynotes-timeseries.svg')